# XAI — Cross-Dataset Explainability (WESAD → WDD / Anxolotl)

The models trained on WESAD are evaluated on two external datasets: **WDD** (Wrist Dataset of Depression) and **Anxolotl** (naturalistic anxiety). This notebook applies SHAP to those cross-dataset predictions to answer:

- Which features drive predictions on WDD and Anxolotl?
- Do the same features matter as in WESAD, or does the model rely on different cues when generalising?
- Do features with **inverted correlations** (e.g. `acc_wrist_min` is positively correlated with stress in WESAD but negatively in WDD) still appear as highly important — revealing why some models fail to transfer?

The data preparation mirrors the cross-dataset evaluation exactly: missing features are filled with `NaN` and imputed using the WESAD training statistics.

**Explainers used:** TreeExplainer (LightGBM, Ensemble), DeepExplainer (DANN), GradientExplainer (RNN).

In [ ]:
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import shap
import torch
import torch.nn as nn
from torch.autograd import Function

shap.initjs()

In [ ]:
# ── Load WESAD tabular data ──────────────────────────────────────────────────
with open('/home/matildxi/Secretária/Documentos/Tese/anxolotl/processed_data/wesad/processed_data.pkl', 'rb') as f:
    data_wesad = pickle.load(f)
X_wesad = data_wesad['X']
y_wesad = data_wesad['y']

# ── Load WDD data ────────────────────────────────────────────────────────────
with open('/home/matildxi/Secretária/Documentos/Tese/anxolotl/processed_data/wdd/processed_data.pkl', 'rb') as f:
    data_wdd = pickle.load(f)
X_wdd = data_wdd['X']
y_wdd = data_wdd['y']

with open('/home/matildxi/Secretária/Documentos/Tese/anxolotl/processed_data/wdd/processed_data_sequences.pkl', 'rb') as f:
    data_wdd_seq = pickle.load(f)
X_wdd_seq = data_wdd_seq['X_seq']   # (5595, 7, 30)
y_wdd_seq = data_wdd_seq['y_seq']

# ── Load Anxolotl data ───────────────────────────────────────────────────────
with open('/home/matildxi/Secretária/Documentos/Tese/anxolotl/processed_data/anxolotl/processed_data_anxolotl.pkl', 'rb') as f:
    data_anx = pickle.load(f)
X_anx = data_anx['X']
y_anx = data_anx['y']

with open('/home/matildxi/Secretária/Documentos/Tese/anxolotl/processed_data/anxolotl/processed_data_anxolotl_seq.pkl', 'rb') as f:
    data_anx_seq = pickle.load(f)
X_anx_seq = data_anx_seq['X_seq']   # (20807, 5, 60)
y_anx_seq = data_anx_seq['y_seq']

print(f'WESAD:    {X_wesad.shape if hasattr(X_wesad,"shape") else len(X_wesad)} samples')
print(f'WDD:      {X_wdd.shape} tabular | {X_wdd_seq.shape} sequences')
print(f'Anxolotl: {X_anx.shape} tabular | {X_anx_seq.shape} sequences')

In [ ]:
# ── Load model checkpoints ───────────────────────────────────────────────────
with open('/home/matildxi/Secretária/Documentos/Tese/anxolotl/models_data/lgbm_wrist_final.pkl', 'rb') as f:
    lgbm_data = pickle.load(f)

with open('/home/matildxi/Secretária/Documentos/Tese/anxolotl/models_data/ensemble_wrist_final.pkl', 'rb') as f:
    ensemble_data = pickle.load(f)

dann_ckpt = torch.load('/home/matildxi/Secretária/Documentos/Tese/anxolotl/models_data/dann_wrist_final.pt', map_location='cpu', weights_only=False)
rnn_ckpt  = torch.load('/home/matildxi/Secretária/Documentos/Tese/anxolotl/models_data/rnn_wrist_final.pt',  map_location='cpu', weights_only=False)

# ── RNN model ────────────────────────────────────────────────────────────────
class RNN1D(nn.Module):
    def __init__(self, input_size=5, hidden_size=128, num_layers=2, num_classes=2):
        super().__init__()
        self.rnn = nn.GRU(input_size=input_size, hidden_size=hidden_size,
                          num_layers=num_layers, batch_first=True,
                          dropout=0.3, bidirectional=True)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size * 2, 128), nn.ReLU(),
            nn.Dropout(0.3), nn.Linear(128, num_classes)
        )
    def forward(self, x):
        x = x.permute(0, 2, 1)
        out, _ = self.rnn(x)
        return self.classifier(out[:, -1, :])

rnn_model = RNN1D(input_size=5)
rnn_model.load_state_dict(rnn_ckpt['model_state_dict'])
rnn_model.eval()

# ── DANN model (stress branch only, no GRL) ──────────────────────────────────
class FeatureExtractor(nn.Module):
    def __init__(self, n):
        super().__init__()
        self.rede = nn.Sequential(nn.Linear(n, 128), nn.ReLU(), nn.Linear(128, 64), nn.ReLU())
    def forward(self, x): return self.rede(x)

class StressClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.rede = nn.Sequential(nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 2))
    def forward(self, x): return self.rede(x)

class DANNClassifier(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.extractor  = FeatureExtractor(n_features)
        self.stress_cls = StressClassifier()
    def forward(self, x): return self.stress_cls(self.extractor(x))

dann_cls = DANNClassifier(dann_ckpt['n_features'])
sub_state = {k: v for k, v in dann_ckpt['model_state_dict'].items()
             if k.startswith('extractor.') or k.startswith('stress_cls.')}
dann_cls.load_state_dict(sub_state)
dann_cls.eval()

dann_feat_cols = dann_ckpt['feature_cols']
dann_top_k     = dann_ckpt['top_k_idx']
dann_sel_cols  = [dann_feat_cols[i] for i in dann_top_k]
print(f'DANN features ({len(dann_sel_cols)}): {dann_sel_cols[:5]}...')
print(f'RNN selected_idx: {rnn_ckpt["selected_idx"]}')

In [ ]:
# ── Data preparation helpers (mirror cross-dataset eval pipeline) ─────────────

def prep_tabular(model_data, X_df_raw):
    """Apply WESAD scaler pipeline: rename acc_energy → acc_wrist, fill missing as NaN."""
    cols = model_data['feature_cols']
    X_df = X_df_raw.copy()
    X_df = X_df.rename(columns={c: c.replace('acc_energy_', 'acc_wrist_')
                                  for c in X_df.columns if c.startswith('acc_energy_')})
    for col in cols:
        if col not in X_df.columns:
            X_df[col] = np.nan
    X_sel = X_df[cols]
    return model_data['imputer'].transform(model_data['scaler'].transform(X_sel))

def prep_dann(X_df_raw):
    """Apply DANN pipeline: WESAD scaler → top_k features."""
    fc   = dann_ckpt['feature_cols']
    X_df = X_df_raw.copy()
    X_df = X_df.rename(columns={c: c.replace('acc_energy_', 'acc_wrist_')
                                  for c in X_df.columns if c.startswith('acc_energy_')})
    for col in fc:
        if col not in X_df.columns:
            X_df[col] = np.nan
    X_sc = dann_ckpt['imputer'].transform(dann_ckpt['scaler'].transform(X_df[fc]))
    return X_sc[:, dann_ckpt['top_k_idx']].astype(np.float32)

def shap_bar_grouped(svs_dict, cols, title, top_n=15, figsize=(14, 6)):
    """
    Side-by-side bar chart comparing mean|SHAP| across datasets.
    svs_dict: {'WESAD': sv_array, 'WDD': sv_array, 'Anxolotl': sv_array}
    """
    # Compute mean |SHAP| per feature for each dataset (align to same feature list)
    imp = {}
    for ds, sv in svs_dict.items():
        imp[ds] = np.abs(sv).mean(axis=0)

    # Rank features by WESAD importance
    ref = list(svs_dict.keys())[0]
    top_idx = np.argsort(imp[ref])[::-1][:top_n]
    feat_names = np.array(cols)[top_idx]

    ds_list = list(svs_dict.keys())
    x = np.arange(len(feat_names))
    w = 0.8 / len(ds_list)
    colors = ['#2196F3', '#FF9800', '#9C27B0']

    fig, ax = plt.subplots(figsize=figsize)
    for i, (ds, color) in enumerate(zip(ds_list, colors)):
        vals = imp[ds][top_idx]
        ax.bar(x + i * w - w * (len(ds_list) - 1) / 2, vals, w, label=ds,
               color=color, alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels(feat_names, rotation=45, ha='right', fontsize=9)
    ax.set_ylabel('Mean |SHAP value|')
    ax.set_title(title)
    ax.legend()
    plt.tight_layout()
    plt.show()

def shap_signal_groups(svs_dict, cols, title):
    """Grouped signal importance comparison across datasets."""
    groups = {
        'ACC':   [c for c in cols if c.startswith('acc')],
        'EDA':   [c for c in cols if c.startswith('eda')],
        'TEMP':  [c for c in cols if c.startswith('temp')],
        'HR':    [c for c in cols if c.startswith('hr')],
        'IBI':   [c for c in cols if c.startswith('ibi')],
        'RMSSD': [c for c in cols if c.startswith('rmssd')],
        'SDNN':  [c for c in cols if c.startswith('sdnn')],
    }
    ds_list = list(svs_dict.keys())
    x = np.arange(len(groups))
    w = 0.8 / len(ds_list)
    colors = ['#2196F3', '#FF9800', '#9C27B0']

    fig, ax = plt.subplots(figsize=(10, 5))
    for i, (ds, color) in enumerate(zip(ds_list, colors)):
        sv = svs_dict[ds]
        vals = []
        for gcols in groups.values():
            idx = [cols.index(c) for c in gcols if c in cols]
            vals.append(np.abs(sv[:, idx]).mean() if idx else 0.0)
        ax.bar(x + i * w - w * (len(ds_list) - 1) / 2, vals, w,
               label=ds, color=color, alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels(list(groups.keys()))
    ax.set_ylabel('Mean |SHAP value|')
    ax.set_title(title)
    ax.legend()
    plt.tight_layout()
    plt.show()

print('Helpers defined.')

In [ ]:
# ── SHAP — LightGBM: WESAD vs WDD vs Anxolotl ────────────────────────────────
np.random.seed(42)
cols_lgbm = lgbm_data['feature_cols']

X_lgbm_wesad_all = prep_tabular(lgbm_data, X_wesad)
X_lgbm_wdd_all   = prep_tabular(lgbm_data, X_wdd)
X_lgbm_anx_all   = prep_tabular(lgbm_data, X_anx)

N = 1000
idx_w  = np.random.choice(len(X_lgbm_wesad_all), min(N, len(X_lgbm_wesad_all)), replace=False)
idx_wd = np.random.choice(len(X_lgbm_wdd_all),   min(N, len(X_lgbm_wdd_all)),   replace=False)
idx_ax = np.random.choice(len(X_lgbm_anx_all),   min(N, len(X_lgbm_anx_all)),   replace=False)

X_lgbm_wesad = X_lgbm_wesad_all[idx_w]
X_lgbm_wdd   = X_lgbm_wdd_all[idx_wd]
X_lgbm_anx   = X_lgbm_anx_all[idx_ax]

explainer_lgbm = shap.TreeExplainer(lgbm_data['model'])

print('Computing SHAP for LightGBM...')
sv_lgbm_wesad = explainer_lgbm.shap_values(X_lgbm_wesad)
sv_lgbm_wdd   = explainer_lgbm.shap_values(X_lgbm_wdd)
sv_lgbm_anx   = explainer_lgbm.shap_values(X_lgbm_anx)

def get_sv1(sv):
    if isinstance(sv, list): return sv[1]
    if sv.ndim == 3: return sv[:, :, 1]
    return sv

sv_lgbm = {
    'WESAD':    get_sv1(sv_lgbm_wesad),
    'WDD':      get_sv1(sv_lgbm_wdd),
    'Anxolotl': get_sv1(sv_lgbm_anx),
}
print('Done. Shapes:', {k: v.shape for k, v in sv_lgbm.items()})

shap_bar_grouped(sv_lgbm, cols_lgbm,
                 'LightGBM — SHAP Feature Importance (WESAD / WDD / Anxolotl)', top_n=15)
shap_signal_groups(sv_lgbm, cols_lgbm,
                   'LightGBM — SHAP por Sinal (WESAD / WDD / Anxolotl)')


In [ ]:
# ── SHAP — Ensemble: WESAD vs WDD vs Anxolotl ────────────────────────────────
np.random.seed(42)
cols_ens = ensemble_data['feature_cols']

X_ens_wesad_all = prep_tabular(ensemble_data, X_wesad)
X_ens_wdd_all   = prep_tabular(ensemble_data, X_wdd)
X_ens_anx_all   = prep_tabular(ensemble_data, X_anx)

X_ens_wesad = X_ens_wesad_all[idx_w]
X_ens_wdd   = X_ens_wdd_all[idx_wd]
X_ens_anx   = X_ens_anx_all[idx_ax]

explainer_xgb      = shap.TreeExplainer(ensemble_data['model'].named_estimators_['xgb'])
explainer_rf       = shap.TreeExplainer(ensemble_data['model'].named_estimators_['rf'])
explainer_lgbm_ens = shap.TreeExplainer(ensemble_data['model'].named_estimators_['lgbm'])

def ensemble_sv(X):
    return (get_sv1(explainer_xgb.shap_values(X)) +
            get_sv1(explainer_rf.shap_values(X)) +
            get_sv1(explainer_lgbm_ens.shap_values(X))) / 3

print('Computing SHAP for Ensemble...')
sv_ens = {
    'WESAD':    ensemble_sv(X_ens_wesad),
    'WDD':      ensemble_sv(X_ens_wdd),
    'Anxolotl': ensemble_sv(X_ens_anx),
}
print('Done. Shapes:', {k: v.shape for k, v in sv_ens.items()})

shap_bar_grouped(sv_ens, cols_ens,
                 'Ensemble — SHAP Feature Importance (WESAD / WDD / Anxolotl)', top_n=15)
shap_signal_groups(sv_ens, cols_ens,
                   'Ensemble — SHAP por Sinal (WESAD / WDD / Anxolotl)')


In [ ]:
# ── SHAP — DANN: WESAD vs WDD vs Anxolotl ────────────────────────────────────
np.random.seed(42)

X_dann_wesad = prep_dann(X_wesad)
X_dann_wdd   = prep_dann(X_wdd)
X_dann_anx   = prep_dann(X_anx)

y_w = np.array(y_wesad)
bg_idx   = np.concatenate([np.where(y_w == 0)[0][:100], np.where(y_w == 1)[0][:100]])
X_bg_dann = torch.tensor(X_dann_wesad[bg_idx])

ts_wesad  = np.concatenate([np.where(y_w == 0)[0][100:200], np.where(y_w == 1)[0][100:200]])
X_ts_wesad = torch.tensor(X_dann_wesad[ts_wesad])

n_samp = 200
X_ts_wdd = torch.tensor(X_dann_wdd[np.random.choice(len(X_dann_wdd), min(n_samp, len(X_dann_wdd)), replace=False)])
X_ts_anx = torch.tensor(X_dann_anx[np.random.choice(len(X_dann_anx), min(n_samp, len(X_dann_anx)), replace=False)])

print('Computing SHAP for DANN...')
explainer_dann_cd = shap.DeepExplainer(dann_cls, X_bg_dann)

def dann_sv(X_ts):
    sv_all = explainer_dann_cd.shap_values(X_ts)
    if isinstance(sv_all, list):   return sv_all[1]
    if sv_all.ndim == 3:           return sv_all[..., 1]
    return sv_all

sv_dann = {
    'WESAD':    dann_sv(X_ts_wesad),
    'WDD':      dann_sv(X_ts_wdd),
    'Anxolotl': dann_sv(X_ts_anx),
}
print('Done. Shapes:', {k: v.shape for k, v in sv_dann.items()})

shap_bar_grouped(sv_dann, dann_sel_cols,
                 'DANN — SHAP Feature Importance (WESAD / WDD / Anxolotl)', top_n=15)
shap_signal_groups(sv_dann, dann_sel_cols,
                   'DANN — SHAP por Sinal (WESAD / WDD / Anxolotl)')


In [ ]:
# ── SHAP — RNN: WDD vs Anxolotl ──────────────────────────────────────────────
# WDD sequences:     (5595, 7, 30) — channel_idx [0,3,4,5,6] → (n, 5, 30)
# Anxolotl sequences:(20807, 5, 60) — all 5 channels
# Note: GradientExplainer requires background and test samples to have the same shape.
#   WDD background  = WDD sequences (30 timesteps)
#   Anxolotl bg     = WESAD sequences (60 timesteps, same as training)
np.random.seed(42)

RNN_SEL_IDX_WDD = [0, 3, 4, 5, 6]
selected_idx    = rnn_ckpt['selected_idx']
RNN_CH_NAMES    = [f'ch_{i:02d}' for i in selected_idx]

# Load WESAD sequences for Anxolotl background
wesad_seq_path = '/home/matildxi/Secretária/Documentos/Tese/DANN/wesad/processed_data_sequences.pkl'
with open(wesad_seq_path, 'rb') as f:
    data_wesad_seq = pickle.load(f)
y_rnn_wesad = data_wesad_seq['y_seq']
X_seq_wesad = data_wesad_seq['X_seq'][:, selected_idx, :].astype(np.float32)  # (27543, 5, 60)

# WDD data: 5 channels, 30 timesteps
X_wdd_rnn = X_wdd_seq[:, RNN_SEL_IDX_WDD, :].astype(np.float32)   # (5595, 5, 30)

# Anxolotl data: 5 channels, 60 timesteps
X_anx_rnn = X_anx_seq.astype(np.float32)                            # (20807, 5, 60)

# WDD: background from WDD itself (same 30-timestep shape)
bg_wdd_idx  = np.random.choice(len(X_wdd_rnn), min(200, len(X_wdd_rnn)), replace=False)
X_bg_wdd_r  = torch.tensor(X_wdd_rnn[bg_wdd_idx])                  # (200, 5, 30)
ts_wdd_idx  = np.setdiff1d(np.arange(len(X_wdd_rnn)), bg_wdd_idx)
ts_wdd_idx  = np.random.choice(ts_wdd_idx, min(300, len(ts_wdd_idx)), replace=False)
X_ts_wdd_r  = torch.tensor(X_wdd_rnn[ts_wdd_idx])                  # (300, 5, 30)

# Anxolotl: background from WESAD (same 60-timestep shape as training)
bg0 = np.where(y_rnn_wesad == 0)[0][:100]
bg1 = np.where(y_rnn_wesad == 1)[0][:100]
X_bg_anx_r  = torch.tensor(X_seq_wesad[np.concatenate([bg0, bg1])])  # (200, 5, 60)
idx_anx_r   = np.random.choice(len(X_anx_rnn), min(150, len(X_anx_rnn)), replace=False)
X_ts_anx_r  = torch.tensor(X_anx_rnn[idx_anx_r])                   # (300, 5, 60)

print('Computing SHAP for RNN — WDD...')
explainer_rnn_wdd = shap.GradientExplainer(rnn_model, X_bg_wdd_r)
sv_rnn_wdd_all = explainer_rnn_wdd.shap_values(X_ts_wdd_r)
sv_rnn_wdd = (sv_rnn_wdd_all[..., 1] if (isinstance(sv_rnn_wdd_all, np.ndarray) and sv_rnn_wdd_all.ndim == 4)
              else sv_rnn_wdd_all[1])
print(f'WDD SHAP: {sv_rnn_wdd.shape}')

print('Computing SHAP for RNN — Anxolotl...')
explainer_rnn_anx = shap.GradientExplainer(rnn_model, X_bg_anx_r)
sv_rnn_anx_all = explainer_rnn_anx.shap_values(X_ts_anx_r)
sv_rnn_anx = (sv_rnn_anx_all[..., 1] if (isinstance(sv_rnn_anx_all, np.ndarray) and sv_rnn_anx_all.ndim == 4)
              else sv_rnn_anx_all[1])
print(f'Anxolotl SHAP: {sv_rnn_anx.shape}')

# ─── Channel importance: WDD vs Anxolotl (normalised) ───
ch_imp_wdd = np.abs(sv_rnn_wdd).mean(axis=(0, 2))
ch_imp_anx = np.abs(sv_rnn_anx).mean(axis=(0, 2))
ch_imp_wdd_n = ch_imp_wdd / ch_imp_wdd.sum()
ch_imp_anx_n = ch_imp_anx / ch_imp_anx.sum()

x_ax = np.arange(len(RNN_CH_NAMES)); w = 0.35
plt.figure(figsize=(9, 4))
plt.bar(x_ax - w/2, ch_imp_wdd_n, w, label='WDD',      color='#FF9800', alpha=0.85)
plt.bar(x_ax + w/2, ch_imp_anx_n, w, label='Anxolotl', color='#9C27B0', alpha=0.85)
plt.xticks(x_ax, RNN_CH_NAMES)
plt.ylabel('Normalised Mean |SHAP value|')
plt.title('RNN — SHAP Channel Importance: WDD vs Anxolotl')
plt.legend(); plt.tight_layout(); plt.show()

# ─── Heatmaps ───
for ds_name, sv in [('WDD', sv_rnn_wdd), ('Anxolotl', sv_rnn_anx)]:
    heat = np.abs(sv).mean(axis=0)   # (5, T)
    plt.figure(figsize=(14, 4))
    plt.imshow(heat, aspect='auto', cmap='YlOrRd', interpolation='nearest')
    plt.colorbar(label='Mean |SHAP|')
    plt.yticks(range(len(RNN_CH_NAMES)), RNN_CH_NAMES)
    plt.xlabel('Timestep')
    plt.title(f'RNN SHAP Heatmap Canal × Timestep — {ds_name}')
    plt.tight_layout(); plt.show()


In [ ]:
# ── Summary: Signal Group Importance — All Models × All Datasets ──────────────
# One figure with 4 subplots (LightGBM, Ensemble, DANN + bar for each dataset)

groups_order = ['ACC', 'EDA', 'TEMP', 'HR', 'IBI', 'RMSSD', 'SDNN']

def signal_imp(sv, cols):
    groups = {
        'ACC':   [c for c in cols if c.startswith('acc')],
        'EDA':   [c for c in cols if c.startswith('eda')],
        'TEMP':  [c for c in cols if c.startswith('temp')],
        'HR':    [c for c in cols if c.startswith('hr')],
        'IBI':   [c for c in cols if c.startswith('ibi')],
        'RMSSD': [c for c in cols if c.startswith('rmssd')],
        'SDNN':  [c for c in cols if c.startswith('sdnn')],
    }
    vals = []
    for g in groups_order:
        idx = [cols.index(c) for c in groups[g] if c in cols]
        vals.append(np.abs(sv[:, idx]).mean() if idx else 0.0)
    # Normalise per dataset so cross-model comparison is fair
    vals = np.array(vals)
    return vals / vals.sum() if vals.sum() > 0 else vals

models = [
    ('LightGBM', sv_lgbm,  cols_lgbm),
    ('Ensemble', sv_ens,   cols_ens),
    ('DANN',     sv_dann,  dann_sel_cols),
]

ds_colors = {'WESAD': '#2196F3', 'WDD': '#FF9800', 'Anxolotl': '#9C27B0'}

fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)
for ax, (model_name, svs_dict, cols) in zip(axes, models):
    x = np.arange(len(groups_order))
    ds_list = list(svs_dict.keys())
    w = 0.8 / len(ds_list)
    for i, ds in enumerate(ds_list):
        vals = signal_imp(svs_dict[ds], cols)
        ax.bar(x + i * w - w * (len(ds_list) - 1) / 2, vals, w,
               label=ds, color=ds_colors[ds], alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels(groups_order, fontsize=9)
    ax.set_title(model_name, fontweight='bold')
    ax.set_ylabel('Normalised Mean |SHAP|')
    if ax == axes[0]: ax.legend(fontsize=8)

plt.suptitle('SHAP Signal Group Importance — Cross-Dataset Comparison', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()